# 06 — Multi-horizon deep-learning models

This notebook reproduces the deep-learning family used to forecast greenhouse air temperature and relative humidity at five temporal resolutions and four forecast horizons.

Four neural architectures are compared: **MLP, LSTM, GRU, and TCN**. Every model receives the same 24-hour `SHT_TIME` sequence and predicts the eight target–horizon combinations simultaneously. Two predefined hyperparameter candidates are evaluated per architecture.

Model selection uses chronological validation data only. Each candidate is evaluated with three independent seeds (`2026`, `2027`, and `2028`), and stability across seeds is retained in the selection tables. After selection, each architecture is refitted on `train + validation`; its test predictions are the average of the three independently fitted models. The test partition is never used to select an architecture, hyperparameter candidate, epoch count, or feature set.

Run notebooks `01`–`05` first. Full execution is computationally intensive and is best run with a GPU-enabled Jupyter environment.


## Dependency note

TensorFlow is required. If the next cell reports that it is missing, run the following command in a separate Jupyter cell, restart the kernel, and then run this notebook from the beginning:

```python
%pip install "tensorflow>=2.16,<2.21" tqdm
```


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import platform
import random
import time
import warnings

if importlib.util.find_spec("tensorflow") is None:
    raise ImportError(
        "TensorFlow is not installed. Run `%pip install \"tensorflow>=2.16,<2.21\"`, "
        "restart the kernel, and execute the notebook again."
    )
if importlib.util.find_spec("tqdm") is None:
    raise ImportError(
        "tqdm is not installed. Run `%pip install tqdm`, restart the kernel, "
        "and execute the notebook again."
    )

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print(f"TensorFlow: {tf.__version__}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "resolutions").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebook 03 first and keep the standard folder structure."
    )


PROJECT_ROOT = find_project_root()
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "effective_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "deep_learning"
MODEL_DIR = PROJECT_ROOT / "models" / "deep_learning"
PREPROCESSOR_DIR = PROJECT_ROOT / "preprocessors" / "deep_learning"
FIGURE_DIR = PROJECT_ROOT / "figures" / "deep_learning"
METADATA_DIR = PROJECT_ROOT / "metadata"

for directory in [RESULTS_DIR, MODEL_DIR, PREPROCESSOR_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")


## Experimental configuration

`EXECUTION_MODE = "full"` reproduces the complete experiment. For a quick structural check, change it to `"smoke_test"`; that mode uses one seed, the first candidate, and two epochs, and its results must not be reported scientifically.

NRMSE is calculated task by task as RMSE divided by the corresponding target standard deviation in the fitting partition. Thus validation and test data never define the normalization scale used for a fitted model.


In [ ]:
EXECUTION_MODE = "full"  # Use "smoke_test" only to check that the pipeline runs.

RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS_MINUTES = [60, 120, 240, 480]
HISTORY_HOURS = 24
FEATURE_SET = "SHT_TIME"
SEQUENCE_FEATURES = [
    "temperature", "relative_humidity",
    "hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos",
]
ARCHITECTURES = ["MLP", "LSTM", "GRU", "TCN"]
SEEDS = [2026, 2027, 2028]
BATCH_SIZE = 128
MAXIMUM_EPOCHS = 80
EARLY_STOPPING_PATIENCE = 10
LEARNING_RATE_PATIENCE = 5

ARCHITECTURE_GRID = {
    "MLP": [
        {"hidden_units": [128, 64], "dropout": 0.2, "learning_rate": 0.001},
        {"hidden_units": [256, 128], "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "LSTM": [
        {"recurrent_units": 64, "dense_units": 32, "dropout": 0.2, "learning_rate": 0.001},
        {"recurrent_units": 96, "dense_units": 48, "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "GRU": [
        {"recurrent_units": 64, "dense_units": 32, "dropout": 0.2, "learning_rate": 0.001},
        {"recurrent_units": 96, "dense_units": 48, "dropout": 0.3, "learning_rate": 0.0005},
    ],
    "TCN": [
        {"filters": 32, "kernel_size": 3, "dilations": [1, 2, 4, 8], "dropout": 0.1, "dense_units": 32, "learning_rate": 0.001},
        {"filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "dropout": 0.2, "dense_units": 48, "learning_rate": 0.0005},
    ],
}

if EXECUTION_MODE == "full":
    ACTIVE_SEEDS = SEEDS
    ACTIVE_GRID = ARCHITECTURE_GRID
    ACTIVE_MAXIMUM_EPOCHS = MAXIMUM_EPOCHS
elif EXECUTION_MODE == "smoke_test":
    ACTIVE_SEEDS = [SEEDS[0]]
    ACTIVE_GRID = {name: candidates[:1] for name, candidates in ARCHITECTURE_GRID.items()}
    ACTIVE_MAXIMUM_EPOCHS = 2
else:
    raise ValueError("EXECUTION_MODE must be 'full' or 'smoke_test'.")

configuration = {
    "execution_mode": EXECUTION_MODE,
    "temporal_resolutions_minutes": RESOLUTIONS,
    "targets": TARGETS,
    "forecast_horizons_minutes": HORIZONS_MINUTES,
    "history_hours": HISTORY_HOURS,
    "feature_set": FEATURE_SET,
    "sequence_features": SEQUENCE_FEATURES,
    "architectures": ARCHITECTURES,
    "architecture_candidates": ARCHITECTURE_GRID,
    "independent_seeds": SEEDS,
    "batch_size": BATCH_SIZE,
    "maximum_epochs": MAXIMUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "learning_rate_reduction": {"factor": 0.5, "patience": LEARNING_RATE_PATIENCE, "minimum_learning_rate": 1e-6},
    "missing_input_policy": "Causal forward fill, followed by medians estimated from the fitting partition only.",
    "scaling_policy": "Input and output StandardScaler objects are fitted on the fitting partition only.",
    "nrmse_definition": "Task RMSE divided by the target standard deviation in the fitting partition.",
    "selection_rule": "Lowest mean validation NRMSE; NRMSE standard deviation and mean validation R2 as secondary criteria.",
    "final_prediction_rule": "Average predictions from independently refitted seed models.",
    "test_policy": "Test is not used for architecture, hyperparameter, epoch, or feature selection.",
    "progress_reporting": "Nested tqdm bars report completed model runs and epochs within the active run.",
}
(METADATA_DIR / "06_deep_learning_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
configuration


## Common forecast origins and sequence construction

The notebook reads the effective origin files already audited in notebooks `03`–`05`. Every architecture receives identical origins, input variables, historical windows, targets, and chronological partitions.

Causal forward filling is performed before window extraction. Any leading or otherwise unresolved missing values are replaced later with medians computed from the fitting partition only.


In [ ]:
EXPECTED_COUNTS = {
    4: {"train": 1053, "validation": 1446, "test": 2742},
    12: {"train": 104, "validation": 135, "test": 378},
    20: {"train": 146, "validation": 239, "test": 556},
    30: {"train": 104, "validation": 142, "test": 350},
    60: {"train": 66, "validation": 58, "test": 211},
}


def load_inputs(resolution_minutes):
    data_path = RESOLUTION_DIR / f"greenhouse_{resolution_minutes}min.csv"
    index_path = INDEX_DIR / f"effective_indices_{resolution_minutes}min.csv"
    if not data_path.exists() or not index_path.exists():
        raise FileNotFoundError(
            f"Missing inputs for {resolution_minutes} min. Run notebooks 03–05 first."
        )
    data = pd.read_csv(data_path, parse_dates=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)
    origins = pd.read_csv(index_path, parse_dates=["origin_timestamp"])
    return data, origins


def build_sequences(data, origins, resolution_minutes):
    prepared = data[SEQUENCE_FEATURES].copy().ffill()
    values = prepared.to_numpy(dtype=np.float32)
    history_steps = HISTORY_HOURS * 60 // resolution_minutes

    sequences = np.empty((len(origins), history_steps, len(SEQUENCE_FEATURES)), dtype=np.float32)
    for row_number, row in enumerate(origins.itertuples(index=False)):
        start = int(row.history_start_index)
        end = int(row.history_end_index) + 1
        window = values[start:end]
        if window.shape != (history_steps, len(SEQUENCE_FEATURES)):
            raise ValueError(
                f"Unexpected window shape at {resolution_minutes} min origin {row.origin_index}: {window.shape}"
            )
        sequences[row_number] = window

    output_columns = []
    outputs = []
    for target in TARGETS:
        for horizon in HORIZONS_MINUTES:
            output_columns.append(f"{target}__h{horizon}")
            positions = origins[f"target_index_h{horizon}"].astype(int).to_numpy()
            outputs.append(data.loc[positions, target].to_numpy(dtype=np.float32))
    targets = np.column_stack(outputs).astype(np.float32)
    return sequences, targets, output_columns


datasets = {}
effective_origins = {}
sequence_matrices = {}
target_matrices = {}
output_names = None
sample_rows = []

for resolution in RESOLUTIONS:
    data, origins = load_inputs(resolution)
    X, Y, names = build_sequences(data, origins, resolution)
    datasets[resolution] = data
    effective_origins[resolution] = origins
    sequence_matrices[resolution] = X
    target_matrices[resolution] = Y
    output_names = names

    counts = origins["split"].value_counts().to_dict()
    row = {
        "resolution": f"{resolution}min",
        "rows": len(data),
        "history_steps": X.shape[1],
        "sequence_features": X.shape[2],
        "outputs": Y.shape[1],
        "total_origins": len(origins),
        "train": int(counts.get("train", 0)),
        "validation": int(counts.get("validation", 0)),
        "test": int(counts.get("test", 0)),
    }
    row["matches_historical_audit"] = all(
        row[split] == expected for split, expected in EXPECTED_COUNTS[resolution].items()
    )
    sample_rows.append(row)

sample_audit = pd.DataFrame(sample_rows)
sample_audit.to_csv(RESULTS_DIR / "01_input_sample_audit.csv", index=False)
feature_map = pd.DataFrame({
    "resolution": [f"{resolution}min" for resolution in RESOLUTIONS],
    "feature_set": [FEATURE_SET] * len(RESOLUTIONS),
    "sequence_features": [";".join(SEQUENCE_FEATURES)] * len(RESOLUTIONS),
})
feature_map.to_csv(RESULTS_DIR / "02_feature_map.csv", index=False)
display(sample_audit)
assert sample_audit["matches_historical_audit"].all(), (
    "Effective sample counts differ from the audited experiment. Investigate upstream inputs first."
)


## Training-only preprocessing

The input scaler is fitted across all time steps belonging to fitting samples. The output scaler is fitted across the eight simultaneous targets. Both scalers, together with the training medians and column order, are serialized for the selected final configurations.


In [ ]:
def split_mask(origins, split):
    return origins["split"].eq(split).to_numpy()


def fit_preprocessor(X, Y, fit_mask):
    fitting_values = X[fit_mask].reshape(-1, X.shape[-1])
    medians = np.nanmedian(fitting_values, axis=0)
    if np.isnan(medians).any():
        missing_features = np.asarray(SEQUENCE_FEATURES)[np.isnan(medians)].tolist()
        raise ValueError(f"No finite fitting values for features: {missing_features}")
    fitting_values = np.where(np.isnan(fitting_values), medians, fitting_values)

    input_scaler = StandardScaler().fit(fitting_values)
    output_scaler = StandardScaler().fit(Y[fit_mask])
    return {
        "input_medians": medians.astype(np.float32),
        "input_scaler": input_scaler,
        "output_scaler": output_scaler,
        "sequence_features": SEQUENCE_FEATURES,
        "output_names": output_names,
        "history_hours": HISTORY_HOURS,
    }


def transform_inputs(X, preprocessor):
    medians = preprocessor["input_medians"]
    imputed = np.where(np.isnan(X), medians[None, None, :], X)
    flat = imputed.reshape(-1, imputed.shape[-1])
    scaled = preprocessor["input_scaler"].transform(flat)
    return scaled.reshape(imputed.shape).astype(np.float32)


def transform_outputs(Y, preprocessor):
    return preprocessor["output_scaler"].transform(Y).astype(np.float32)


def inverse_outputs(Y_scaled, preprocessor):
    return preprocessor["output_scaler"].inverse_transform(Y_scaled).astype(np.float32)


training_preprocessors = {}
scaled_training_data = {}
for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    preprocessor = fit_preprocessor(
        sequence_matrices[resolution], target_matrices[resolution], train
    )
    training_preprocessors[resolution] = preprocessor
    scaled_training_data[resolution] = (
        transform_inputs(sequence_matrices[resolution], preprocessor),
        transform_outputs(target_matrices[resolution], preprocessor),
    )


## Neural architectures

The MLP flattens the sequence. LSTM and GRU preserve the recurrent structure. The TCN uses causal residual convolutional blocks with dilation rates 1, 2, 4, and 8. All networks optimize mean squared error with Adam and produce eight linear outputs.


In [ ]:
def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def tcn_residual_block(x, filters, kernel_size, dilation, dropout):
    residual = x
    x = layers.Conv1D(
        filters, kernel_size, padding="causal", dilation_rate=dilation,
        activation="relu", kernel_initializer="he_normal",
    )(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(
        filters, kernel_size, padding="causal", dilation_rate=dilation,
        activation=None, kernel_initializer="he_normal",
    )(x)
    if residual.shape[-1] != filters:
        residual = layers.Conv1D(filters, 1, padding="same")(residual)
    x = layers.Add()([x, residual])
    return layers.Activation("relu")(x)


def build_model(architecture, input_shape, output_size, parameters, seed):
    keras.backend.clear_session()
    reset_seed(seed)
    inputs = keras.Input(shape=input_shape, name="history_sequence")

    if architecture == "MLP":
        x = layers.Flatten()(inputs)
        for units in parameters["hidden_units"]:
            x = layers.Dense(units, activation="relu")(x)
            x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "LSTM":
        x = layers.LSTM(
            parameters["recurrent_units"], dropout=parameters["dropout"]
        )(inputs)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "GRU":
        x = layers.GRU(
            parameters["recurrent_units"], dropout=parameters["dropout"]
        )(inputs)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    elif architecture == "TCN":
        x = inputs
        for dilation in parameters["dilations"]:
            x = tcn_residual_block(
                x, parameters["filters"], parameters["kernel_size"],
                dilation, parameters["dropout"],
            )
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(parameters["dense_units"], activation="relu")(x)
        x = layers.Dropout(parameters["dropout"])(x)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    outputs = layers.Dense(output_size, activation="linear", name="multihorizon_outputs")(x)
    model = keras.Model(inputs=inputs, outputs=outputs, name=architecture.lower())
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=parameters["learning_rate"]),
        loss="mse",
    )
    return model


def validation_callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, mode="min",
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=LEARNING_RATE_PATIENCE,
            min_lr=1e-6, mode="min",
        ),
    ]


class EpochProgress(keras.callbacks.Callback):
    # Compact per-epoch progress bar compatible with silent Keras training.

    def __init__(self, total_epochs, description, position=1):
        super().__init__()
        self.progress = tqdm(
            total=int(total_epochs),
            desc=description,
            unit="epoch",
            leave=False,
            position=position,
            dynamic_ncols=True,
        )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        postfix = {}
        if "loss" in logs:
            postfix["loss"] = f"{float(logs['loss']):.4f}"
        if "val_loss" in logs:
            postfix["val_loss"] = f"{float(logs['val_loss']):.4f}"
        if postfix:
            self.progress.set_postfix(postfix, refresh=False)
        self.progress.update(1)

    def on_train_end(self, logs=None):
        self.progress.close()


In [ ]:
def task_metrics(y_true, y_pred, normalization_scale):
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            observed = y_true[:, output_index]
            predicted = y_pred[:, output_index]
            rmse = float(np.sqrt(mean_squared_error(observed, predicted)))
            scale = float(normalization_scale[output_index])
            rows.append({
                "target": target,
                "horizon_minutes": horizon,
                "n": len(observed),
                "rmse": rmse,
                "nrmse": rmse / scale if scale > 0 else np.nan,
                "r2": float(r2_score(observed, predicted)),
                "mae": float(mean_absolute_error(observed, predicted)),
                "bias": float(np.mean(predicted - observed)),
            })
    return pd.DataFrame(rows)


def prediction_rows(origins, y_true, y_pred, resolution, architecture, split):
    selected = origins.loc[origins["split"].eq(split)].reset_index(drop=True)
    rows = []
    for target_index, target in enumerate(TARGETS):
        for horizon_index, horizon in enumerate(HORIZONS_MINUTES):
            output_index = target_index * len(HORIZONS_MINUTES) + horizon_index
            rows.append(pd.DataFrame({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "architecture": architecture,
                "feature_set": FEATURE_SET,
                "split": split,
                "origin_index": selected["origin_index"],
                "origin_timestamp": selected["origin_timestamp"],
                "target": target,
                "horizon_minutes": horizon,
                "observed": y_true[:, output_index],
                "predicted": y_pred[:, output_index],
            }))
    return pd.concat(rows, ignore_index=True)


## Phase A — candidate evaluation across independent seeds

Each candidate is trained separately with every active seed. Candidate ranking uses mean validation NRMSE, followed by its standard deviation across seeds and mean validation $R^2$. The best epoch is determined independently for every seed by validation loss.


In [ ]:
seed_validation_metric_frames = []
seed_run_rows = []
history_rows = []
validation_prediction_cache = {}

phase_a_total_runs = (
    len(RESOLUTIONS)
    * len(ACTIVE_SEEDS)
    * sum(len(ACTIVE_GRID[architecture]) for architecture in ARCHITECTURES)
)
phase_a_progress = tqdm(
    total=phase_a_total_runs,
    desc="Phase A — candidate models",
    unit="model",
    position=0,
    dynamic_ncols=True,
)

for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    X_scaled, Y_scaled = scaled_training_data[resolution]
    Y = target_matrices[resolution]
    preprocessor = training_preprocessors[resolution]
    normalization_scale = np.std(Y[train], axis=0, ddof=0)

    for architecture in ARCHITECTURES:
        for candidate_id, parameters in enumerate(ACTIVE_GRID[architecture], start=1):
            for seed in ACTIVE_SEEDS:
                run_label = (
                    f"{resolution} min | {architecture} | "
                    f"candidate {candidate_id} | seed {seed}"
                )
                model = build_model(
                    architecture, X_scaled.shape[1:], Y_scaled.shape[1], parameters, seed
                )
                start = time.perf_counter()
                history = model.fit(
                    X_scaled[train], Y_scaled[train],
                    validation_data=(X_scaled[validation], Y_scaled[validation]),
                    epochs=ACTIVE_MAXIMUM_EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=[
                        *validation_callbacks(),
                        EpochProgress(
                            ACTIVE_MAXIMUM_EPOCHS,
                            run_label,
                            position=1,
                        ),
                    ],
                    verbose=0,
                    shuffle=True,
                )
                fit_seconds = time.perf_counter() - start
                best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
                phase_a_progress.set_postfix({
                    "resolution": f"{resolution} min",
                    "architecture": architecture,
                    "candidate": candidate_id,
                    "seed": seed,
                    "epochs": len(history.history["loss"]),
                    "last_min": f"{fit_seconds / 60:.1f}",
                }, refresh=False)
                phase_a_progress.update(1)

                start = time.perf_counter()
                prediction_scaled = model.predict(
                    X_scaled[validation], batch_size=BATCH_SIZE, verbose=0
                )
                inference_seconds = time.perf_counter() - start
                prediction = inverse_outputs(prediction_scaled, preprocessor)
                validation_prediction_cache[(
                    resolution, architecture, candidate_id, seed
                )] = prediction
                metrics = task_metrics(Y[validation], prediction, normalization_scale).assign(
                    resolution=f"{resolution}min",
                    resolution_minutes=resolution,
                    architecture=architecture,
                    feature_set=FEATURE_SET,
                    candidate_id=candidate_id,
                    seed=seed,
                    split="validation",
                    best_epoch=best_epoch,
                    fit_seconds=fit_seconds,
                    inference_seconds=inference_seconds,
                    parameters=json.dumps(parameters, sort_keys=True),
                )
                seed_validation_metric_frames.append(metrics)
                seed_run_rows.append({
                    "resolution": f"{resolution}min",
                    "resolution_minutes": resolution,
                    "architecture": architecture,
                    "candidate_id": candidate_id,
                    "seed": seed,
                    "parameters": json.dumps(parameters, sort_keys=True),
                    "best_epoch": best_epoch,
                    "fit_seconds": fit_seconds,
                    "inference_seconds": inference_seconds,
                    "mean_normalized_rmse": float(metrics["nrmse"].mean()),
                    "mean_rmse": float(metrics["rmse"].mean()),
                    "mean_r2": float(metrics["r2"].mean()),
                    "minimum_r2": float(metrics["r2"].min()),
                })
                for epoch, (loss, val_loss) in enumerate(
                    zip(history.history["loss"], history.history["val_loss"]), start=1
                ):
                    history_rows.append({
                        "resolution": f"{resolution}min",
                        "architecture": architecture,
                        "candidate_id": candidate_id,
                        "seed": seed,
                        "training_scope": "train_with_validation_monitoring",
                        "epoch": epoch,
                        "loss": loss,
                        "val_loss": val_loss,
                    })
                del model
                keras.backend.clear_session()

phase_a_progress.close()
seed_validation_metrics = pd.concat(seed_validation_metric_frames, ignore_index=True)
seed_run_summary = pd.DataFrame(seed_run_rows)
training_histories = pd.DataFrame(history_rows)
seed_validation_metrics.to_csv(RESULTS_DIR / "03_seed_validation_metrics.csv", index=False)
training_histories.to_csv(RESULTS_DIR / "07_training_histories.csv", index=False)

candidate_summary = (
    seed_run_summary.groupby(
        ["resolution", "resolution_minutes", "architecture", "candidate_id", "parameters"],
        as_index=False,
    )
    .agg(
        mean_normalized_rmse=("mean_normalized_rmse", "mean"),
        sd_normalized_rmse=("mean_normalized_rmse", "std"),
        mean_rmse=("mean_rmse", "mean"),
        sd_rmse=("mean_rmse", "std"),
        mean_r2=("mean_r2", "mean"),
        sd_r2=("mean_r2", "std"),
        minimum_r2=("minimum_r2", "min"),
        median_best_epoch=("best_epoch", "median"),
        mean_fit_seconds=("fit_seconds", "mean"),
        mean_inference_seconds=("inference_seconds", "mean"),
        independent_runs=("seed", "nunique"),
    )
)
candidate_summary[["sd_normalized_rmse", "sd_rmse", "sd_r2"]] = candidate_summary[
    ["sd_normalized_rmse", "sd_rmse", "sd_r2"]
].fillna(0.0)
candidate_summary.to_csv(RESULTS_DIR / "04_hyperparameter_tuning.csv", index=False)
display(candidate_summary)


## Architecture and hyperparameter selection

The best candidate is first selected within each architecture. The four retained architectures are then ranked within each temporal resolution. Stability is an explicit secondary criterion rather than an informal visual judgment.


In [ ]:
best_candidate_rows = []
for (resolution, architecture), subset in candidate_summary.groupby(
    ["resolution_minutes", "architecture"], sort=False
):
    winner = subset.sort_values(
        ["mean_normalized_rmse", "sd_normalized_rmse", "mean_r2"],
        ascending=[True, True, False],
    ).iloc[0]
    best_candidate_rows.append(winner.to_dict())

architecture_summary = pd.DataFrame(best_candidate_rows)
architecture_summary.to_csv(RESULTS_DIR / "05_architecture_validation_summary.csv", index=False)

selected_architecture_rows = []
for resolution, subset in architecture_summary.groupby("resolution_minutes", sort=False):
    winner = subset.sort_values(
        ["mean_normalized_rmse", "sd_normalized_rmse", "mean_r2"],
        ascending=[True, True, False],
    ).iloc[0]
    row = winner.to_dict()
    row["selection_rule"] = (
        "lowest mean validation NRMSE; NRMSE stability and mean validation R2 as secondary criteria"
    )
    selected_architecture_rows.append(row)

selected_architectures = pd.DataFrame(selected_architecture_rows)
selected_architectures.to_csv(
    RESULTS_DIR / "06_selected_architecture_by_resolution.csv", index=False
)
display(architecture_summary.sort_values(["resolution_minutes", "mean_normalized_rmse"]))
display(selected_architectures)

validation_ensemble_metric_frames = []
validation_prediction_frames = []
for selected_candidate in architecture_summary.itertuples(index=False):
    resolution = int(selected_candidate.resolution_minutes)
    architecture = selected_candidate.architecture
    candidate_id = int(selected_candidate.candidate_id)
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    Y = target_matrices[resolution]
    ensemble_prediction = np.mean([
        validation_prediction_cache[(resolution, architecture, candidate_id, seed)]
        for seed in ACTIVE_SEEDS
    ], axis=0)
    metrics = task_metrics(
        Y[validation], ensemble_prediction, np.std(Y[train], axis=0, ddof=0)
    ).assign(
        resolution=f"{resolution}min",
        resolution_minutes=resolution,
        architecture=architecture,
        feature_set=FEATURE_SET,
        candidate_id=candidate_id,
        split="validation",
        epochs=max(1, int(round(selected_candidate.median_best_epoch))),
        independent_runs=len(ACTIVE_SEEDS),
    )
    validation_ensemble_metric_frames.append(metrics)
    validation_prediction_frames.append(prediction_rows(
        origins, Y[validation], ensemble_prediction,
        resolution, architecture, "validation"
    ))

validation_ensemble_metrics = pd.concat(
    validation_ensemble_metric_frames, ignore_index=True
)
validation_predictions = pd.concat(validation_prediction_frames, ignore_index=True)


## Final refitting and independent test evaluation

For every retained architecture, the epoch count is the median best epoch across its validation runs. Each seed is then refitted on `train + validation` for that fixed number of epochs. Test predictions are averaged across seeds before the primary metrics are calculated.

Only the three seed models belonging to the validation-selected architecture at each resolution are serialized. Models from non-selected architectures remain reproducible from the notebook but are not retained, which keeps the repository size manageable.


In [ ]:
test_metric_frames = []
seed_test_metric_frames = []
prediction_frames = []
final_run_rows = []
final_history_rows = []
saved_model_rows = []

final_total_runs = len(RESOLUTIONS) * len(ARCHITECTURES) * len(ACTIVE_SEEDS)
final_progress = tqdm(
    total=final_total_runs,
    desc="Final refit — train + validation",
    unit="model",
    position=0,
    dynamic_ncols=True,
)

for resolution in RESOLUTIONS:
    origins = effective_origins[resolution]
    train = split_mask(origins, "train")
    validation = split_mask(origins, "validation")
    test = split_mask(origins, "test")
    fit_scope = train | validation
    X = sequence_matrices[resolution]
    Y = target_matrices[resolution]

    final_preprocessor = fit_preprocessor(X, Y, fit_scope)
    X_scaled = transform_inputs(X, final_preprocessor)
    Y_scaled = transform_outputs(Y, final_preprocessor)
    normalization_scale = np.std(Y[fit_scope], axis=0, ddof=0)

    selected_architecture = selected_architectures.loc[
        selected_architectures["resolution_minutes"] == resolution, "architecture"
    ].iloc[0]

    for architecture in ARCHITECTURES:
        selected_candidate = architecture_summary.loc[
            (architecture_summary["resolution_minutes"] == resolution)
            & (architecture_summary["architecture"] == architecture)
        ].iloc[0]
        candidate_id = int(selected_candidate["candidate_id"])
        parameters = json.loads(selected_candidate["parameters"])
        epochs = max(1, int(round(selected_candidate["median_best_epoch"])))
        seed_predictions = []

        for seed in ACTIVE_SEEDS:
            run_label = (
                f"{resolution} min | {architecture} | "
                f"candidate {candidate_id} | seed {seed}"
            )
            model = build_model(
                architecture, X_scaled.shape[1:], Y_scaled.shape[1], parameters, seed
            )
            start = time.perf_counter()
            history = model.fit(
                X_scaled[fit_scope], Y_scaled[fit_scope],
                epochs=epochs,
                batch_size=BATCH_SIZE,
                callbacks=[EpochProgress(epochs, run_label, position=1)],
                verbose=0,
                shuffle=True,
            )
            fit_seconds = time.perf_counter() - start
            final_progress.set_postfix({
                "resolution": f"{resolution} min",
                "architecture": architecture,
                "candidate": candidate_id,
                "seed": seed,
                "epochs": len(history.history["loss"]),
                "last_min": f"{fit_seconds / 60:.1f}",
            }, refresh=False)
            final_progress.update(1)
            start = time.perf_counter()
            prediction_scaled = model.predict(X_scaled[test], batch_size=BATCH_SIZE, verbose=0)
            inference_seconds = time.perf_counter() - start
            prediction = inverse_outputs(prediction_scaled, final_preprocessor)
            seed_predictions.append(prediction)

            seed_metrics = task_metrics(Y[test], prediction, normalization_scale).assign(
                resolution=f"{resolution}min",
                resolution_minutes=resolution,
                architecture=architecture,
                feature_set=FEATURE_SET,
                candidate_id=candidate_id,
                seed=seed,
                split="test",
                epochs=epochs,
                fit_seconds=fit_seconds,
                inference_seconds=inference_seconds,
            )
            seed_test_metric_frames.append(seed_metrics)
            final_run_rows.append({
                "resolution": f"{resolution}min",
                "resolution_minutes": resolution,
                "architecture": architecture,
                "candidate_id": candidate_id,
                "seed": seed,
                "epochs": epochs,
                "fit_seconds": fit_seconds,
                "inference_seconds": inference_seconds,
            })
            for epoch, loss in enumerate(history.history["loss"], start=1):
                final_history_rows.append({
                    "resolution": f"{resolution}min",
                    "architecture": architecture,
                    "candidate_id": candidate_id,
                    "seed": seed,
                    "training_scope": "train_plus_validation_fixed_epochs",
                    "epoch": epoch,
                    "loss": loss,
                    "val_loss": np.nan,
                })

            if architecture == selected_architecture:
                model_path = (
                    MODEL_DIR / f"{resolution}min" / architecture / FEATURE_SET
                    / f"seed_{seed}.keras"
                )
                model_path.parent.mkdir(parents=True, exist_ok=True)
                model.save(model_path)
                saved_model_rows.append({
                    "resolution": f"{resolution}min",
                    "resolution_minutes": resolution,
                    "architecture": architecture,
                    "feature_set": FEATURE_SET,
                    "seed": seed,
                    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                })
            del model
            keras.backend.clear_session()

        ensemble_prediction = np.mean(seed_predictions, axis=0)
        ensemble_metrics = task_metrics(Y[test], ensemble_prediction, normalization_scale).assign(
            resolution=f"{resolution}min",
            resolution_minutes=resolution,
            architecture=architecture,
            feature_set=FEATURE_SET,
            candidate_id=candidate_id,
            split="test",
            epochs=epochs,
            independent_runs=len(ACTIVE_SEEDS),
        )
        test_metric_frames.append(ensemble_metrics)
        prediction_frames.append(prediction_rows(
            origins, Y[test], ensemble_prediction, resolution, architecture, "test"
        ))

    preprocessor_path = (
        PREPROCESSOR_DIR / f"{resolution}min" / selected_architecture / FEATURE_SET
        / "preprocessing.joblib"
    )
    preprocessor_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(final_preprocessor, preprocessor_path)

final_progress.close()
test_metrics = pd.concat(test_metric_frames, ignore_index=True)
seed_test_metrics = pd.concat(seed_test_metric_frames, ignore_index=True)
test_predictions = pd.concat(prediction_frames, ignore_index=True)
dl_metrics = pd.concat([validation_ensemble_metrics, test_metrics], ignore_index=True)
dl_predictions = pd.concat([validation_predictions, test_predictions], ignore_index=True)
final_runs = pd.DataFrame(final_run_rows)
training_histories = pd.concat(
    [training_histories, pd.DataFrame(final_history_rows)], ignore_index=True
)
saved_models = pd.DataFrame(saved_model_rows)

dl_metrics.to_csv(RESULTS_DIR / "09_dl_metrics.csv", index=False)
dl_predictions.to_csv(RESULTS_DIR / "10_dl_predictions.csv", index=False)
seed_test_metrics.to_csv(RESULTS_DIR / "11_seed_test_metrics.csv", index=False)
training_histories.to_csv(RESULTS_DIR / "07_training_histories.csv", index=False)
saved_models.to_csv(RESULTS_DIR / "12_saved_model_catalog.csv", index=False)


In [ ]:
final_configuration = selected_architectures[[
    "resolution", "resolution_minutes", "architecture", "candidate_id", "parameters",
    "mean_normalized_rmse", "sd_normalized_rmse", "mean_rmse", "mean_r2", "median_best_epoch",
]].copy()
final_configuration["feature_set"] = FEATURE_SET
final_configuration["independent_seeds"] = ";".join(map(str, ACTIVE_SEEDS))
final_configuration["test_used_for_selection"] = False
final_configuration.to_csv(
    RESULTS_DIR / "08_final_configuration_by_resolution.csv", index=False
)

computational_summary = (
    final_runs.groupby(["architecture"], as_index=False)
    .agg(
        mean_fit_seconds=("fit_seconds", "mean"),
        total_fit_seconds=("fit_seconds", "sum"),
        mean_inference_seconds=("inference_seconds", "mean"),
        total_inference_seconds=("inference_seconds", "sum"),
        mean_epochs=("epochs", "mean"),
        independent_runs=("seed", "nunique"),
    )
)
computational_summary.to_csv(RESULTS_DIR / "13_computational_summary.csv", index=False)

HISTORICAL_SELECTION = pd.DataFrame({
    "resolution_minutes": [4, 12, 20, 30, 60],
    "historical_architecture": ["LSTM", "LSTM", "LSTM", "LSTM", "MLP"],
    "historical_candidate_id": [2, 1, 2, 1, 1],
    "historical_feature_set": ["SHT_TIME"] * 5,
})
selection_audit = final_configuration.merge(HISTORICAL_SELECTION, on="resolution_minutes")
selection_audit["matches_historical_architecture"] = (
    selection_audit["architecture"] == selection_audit["historical_architecture"]
)
selection_audit["matches_historical_candidate"] = (
    selection_audit["candidate_id"].astype(int) == selection_audit["historical_candidate_id"]
)
selection_audit["matches_historical_feature_set"] = (
    selection_audit["feature_set"] == selection_audit["historical_feature_set"]
)
selection_audit.to_csv(RESULTS_DIR / "14_selection_audit.csv", index=False)
display(final_configuration)
display(selection_audit[[
    "resolution", "architecture", "candidate_id", "feature_set",
    "matches_historical_architecture", "matches_historical_candidate",
    "matches_historical_feature_set",
]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.lineplot(
    data=architecture_summary,
    x="resolution_minutes", y="mean_normalized_rmse",
    hue="architecture", marker="o", ax=axes[0],
)
axes[0].set_title("Architecture comparison on validation data")
axes[0].set_xlabel("Temporal resolution (min)")
axes[0].set_ylabel("Mean validation NRMSE")

selected_test_metrics = test_metrics.merge(
    final_configuration[["resolution_minutes", "architecture"]],
    on=["resolution_minutes", "architecture"], how="inner",
)
sns.lineplot(
    data=selected_test_metrics,
    x="horizon_minutes", y="rmse",
    hue="target", style="resolution", markers=True, dashes=False, ax=axes[1],
)
axes[1].set_title("Validation-selected deep-learning configurations")
axes[1].set_xlabel("Forecast horizon (min)")
axes[1].set_ylabel("Test RMSE")
axes[1].legend(fontsize=7, ncol=2)

summary_figure = FIGURE_DIR / "06_deep_learning_summary.png"
fig.savefig(summary_figure, dpi=300, bbox_inches="tight")
plt.show()

selected_keys = final_configuration[["resolution", "architecture", "candidate_id"]].copy()
selected_histories = training_histories.merge(
    selected_keys, on=["resolution", "architecture", "candidate_id"], how="inner"
)
selected_histories = selected_histories.loc[
    selected_histories["training_scope"] == "train_with_validation_monitoring"
]
g = sns.relplot(
    data=selected_histories, x="epoch", y="val_loss",
    hue="seed", col="resolution", col_wrap=3,
    kind="line", facet_kws={"sharex": False, "sharey": False},
    height=3.2, aspect=1.2,
)
g.set_axis_labels("Epoch", "Validation loss")
g.set_titles("{col_name}")
training_figure = FIGURE_DIR / "06_selected_training_curves.png"
g.figure.savefig(training_figure, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {summary_figure.relative_to(PROJECT_ROOT)}")
print(f"Saved: {training_figure.relative_to(PROJECT_ROOT)}")


In [ ]:
expected_prediction_rows = sum(
    EXPECTED_COUNTS[resolution]["validation"] + EXPECTED_COUNTS[resolution]["test"]
    for resolution in RESOLUTIONS
) * len(ARCHITECTURES) * len(TARGETS) * len(HORIZONS_MINUTES)
expected_metric_rows = (
    len(RESOLUTIONS) * len(ARCHITECTURES) * 2
    * len(TARGETS) * len(HORIZONS_MINUTES)
)

assert len(dl_predictions) == expected_prediction_rows
assert len(dl_metrics) == expected_metric_rows
assert not dl_predictions.duplicated([
    "resolution", "architecture", "split", "origin_index", "target", "horizon_minutes"
]).any()

output_summary = pd.DataFrame({
    "artifact": [
        "candidate summaries", "seed validation metric rows", "ensemble test metric rows",
        "prediction rows", "serialized selected seed models", "preprocessors", "figures",
    ],
    "count": [
        len(candidate_summary), len(seed_validation_metrics), len(dl_metrics),
        len(dl_predictions), len(saved_models),
        len(list(PREPROCESSOR_DIR.rglob("preprocessing.joblib"))),
        len(list(FIGURE_DIR.glob("*.png"))),
    ],
})
display(output_summary)
print(f"Results written to: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Models written to: {MODEL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Preprocessors written to: {PREPROCESSOR_DIR.relative_to(PROJECT_ROOT)}")


## Reproducibility note

Deep-learning results may show small numerical differences across TensorFlow versions, CPU/GPU kernels, and hardware, even with deterministic seeds. The notebook therefore records both central performance and between-seed variability. Any change in effective origin counts, preprocessing scope, selected configuration, or unusually large seed dispersion should be investigated before comparing the regenerated results with the manuscript.
